# Define mnemonics

In [58]:
import os
import glob

from collections import defaultdict

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import lasio
from dlisio import dlis

## Go to base directory and find all data

In [59]:
place_data = '/Users/leonm4/Desktop/Oil'
os.chdir(place_data)

las_files = glob.glob('**/*.las', recursive=True)
LAS_files = glob.glob('**/*.LAS', recursive=True)
dlis_files = glob.glob('**/*.dlis', recursive=True)
DLIS_files = glob.glob('**/*.DLIS', recursive=True)

## Count all mnemonics of files

In [60]:
mnemonics = defaultdict(int)

In [61]:
for file in las_files + LAS_files:
    las = lasio.read(file, ignore_data=True)
    for curve in las.curves[:]:
        mnemonics[curve.mnemonic] += 1

In [62]:
for file in dlis_files + DLIS_files:
    with dlis.load(file) as files:
        for f in files:
            for frame in f.frames:
                mn = [ch.name for ch in frame.channels]
                for m in mn:
                    mnemonics[m] += 1

In [63]:
copybar = mnemonics.copy()
#copybar.pop("DEPT", None)
sorted_items = sorted(copybar.items(), key=lambda x: x[1], reverse=False)
copybar = dict(sorted_items)

sf = 2
fig = plt.figure(figsize=(30 * sf, 500 * sf))
ax = fig.add_subplot(231)

plt.title(f"Mnemonics of las and dlis files : {len(mnemonics)} curves types")
ax.barh(list(copybar.keys()), list(copybar.values()))
ax.margins(y=0)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_ylabel("Mnemonics")

Text(0, 0.5, 'Mnemonics')

In [64]:
fig.savefig("Mnemonics.png", dpi=300, bbox_inches='tight')#, transparent=True)

In [65]:
mnemonics

defaultdict(int,
            {'DEPT': 550,
             'BS': 192,
             'CS': 18,
             'ICV': 9,
             'IHV': 9,
             'TENS': 18,
             'HTEN': 1,
             'CTEM': 15,
             'GR': 328,
             'NPHI': 276,
             'NPHI_DOL': 3,
             'NPHI_LIM': 3,
             'NPHI_SAN': 3,
             'TNPH': 9,
             'TNPH_DOL': 3,
             'TNPH_LIM': 3,
             'TNPH_SAN': 3,
             'CALI': 264,
             'HDRA': 9,
             'PEF8': 3,
             'PEFZ': 9,
             'RHO8': 3,
             'RHOZ': 9,
             'U8': 3,
             'UZ': 9,
             'DI': 1,
             'MRES': 1,
             'RLA1': 3,
             'RLA2': 3,
             'RLA3': 3,
             'RLA4': 3,
             'RLA5': 3,
             'RT': 19,
             'RXO': 87,
             'SP': 175,
             'DTCO': 3,
             'RXO8': 9,
             'RXOZ': 9,
             'GR:1': 1,
             'TDEP': 80,


# Structure of $\ $ *.dlise files

In [66]:
def dlis_frame_to_dataframe(frame):
    curves = frame.curves()
    data = defaultdict(list)
    for name in curves.dtype.names:
        field = curves[name]
        field_dtype = curves.dtype[name]

        if field_dtype.shape == ():
            data[name] = field
        else:
            if field.ndim == 2:
                flat = np.array([np.array(x).flatten() for x in field])
                for i in range(flat.shape[1]):
                    data[f"{name}_{i+1}"] = flat[:, i]

    df = pd.DataFrame(data)
    return df

In [67]:
with dlis.load(dlis_files[2]) as file:
    print(file.describe())
    for f in file:
        print(f.describe())
        for frame in f.frames:
            print(frame.describe())
            df = dlis_frame_to_dataframe(frame)
            print(df.head(10))

-------------
Physical File
-------------
Number of Logical Files : 1

Description : LogicalFile(CMR_047LUP)
Frames      : 2
Channels    : 297


------------
Logical File
------------
Description : LogicalFile(CMR_047LUP)
Frames      : 2
Channels    : 297

Known objects
--
FILE-HEADER             : 1
PARAMETER               : 324
ORIGIN                  : 1
CALIBRATION             : 288
TOOL                    : 4
CALIBRATION-MEASUREMENT : 101
FRAME                   : 2
CHANNEL                 : 297
EQUIPMENT               : 14
CALIBRATION-COEFFICIENT : 108
PROCESS                 : 1

Unknown objects
--
440-CHANNEL                  : 280
440-OP-CHANNEL               : 297
440-PRESENTATION-DESCRIPTION : 1


-----
Frame
-----
name   : 60B
origin : 41
copy   : 0

Channel indexing
--
Indexed by       : BOREHOLE-DEPTH
Index units      : 0.1 in
Index min        : 0 [0.1 in]
Index max        : 0 [0.1 in]
Direction        : DECREASING
Constant spacing : -60 [0.1 in]
Index channel    : Channe